Load everything from Day 5
scaler has the fitted values(data like the mean and std which is used to transform )

In [1]:
import joblib
import numpy as np
import pandas as pd

# Load the scaled training and test data
X_train_scaled = joblib.load("../models/X_train_scaled.pkl")
X_test_scaled  = joblib.load("../models/X_test_scaled.pkl")
y_train        = joblib.load("../models/y_train.pkl")
y_test         = joblib.load("../models/y_test.pkl")
feature_names  = joblib.load("../models/feature_names.pkl")

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)
print("y_train failure rate:", y_train.mean().round(4))
print("y_test failure rate: ", y_test.mean().round(4))
print("Feature names:", feature_names)

X_train_scaled shape: (8000, 7)
X_test_scaled shape:  (2000, 7)
y_train failure rate: 0.0339
y_test failure rate:  0.034
Feature names: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L', 'Type_M']


What Logistic Regression actually does
Let me explain this literally with math, no stories.
The goal
Given 7 input features (temperature, torque, etc.), output a number between 0 and 1 that represents "probability this machine will fail."

Close to 0 → probably healthy
Close to 1 → probably failing

How it computes that number — in three parts
Part 1: A weighted sum of your features
The model learns 7 numbers called weights (one per feature) plus one extra number called the bias. It computes:
z = (weight_1 × air_temp) 
  + (weight_2 × process_temp) 
  + (weight_3 × rotational_speed) 
  + (weight_4 × torque) 
  + (weight_5 × tool_wear) 
  + (weight_6 × type_L) 
  + (weight_7 × type_M) 
  + bias
That z is just a plain number — it could be anything from negative infinity to positive infinity.
Part 2: Squash z into a probability
We pass z through a function called the sigmoid (also called logistic function):
probability = 1 / (1 + e^(-z))
This function has a nice property: no matter what z is, the output is always between 0 and 1.

If z is very negative (like -5), probability ≈ 0.01
If z is 0, probability = 0.5
If z is very positive (like +5), probability ≈ 0.99

That's where the "logistic" in Logistic Regression comes from.
Part 3: Convert probability to a 0 or 1 prediction
By default, if probability ≥ 0.5, predict 1 (failure). Otherwise predict 0 (healthy). We can change this threshold later — that becomes important.
What "training" means
Training = the algorithm searches for the 8 numbers (7 weights + 1 bias) that make the model's predictions match the training labels as often as possible.
It uses a math technique called gradient descent (you don't need to know this for now — scikit-learn handles it) to find those numbers.
Why "regression" when it's for classification?
Historical naming. Despite the name, Logistic Regression is a classification algorithm. Blame 1950s statisticians.

In [2]:
from sklearn.linear_model import LogisticRegression

# Create the model
model = LogisticRegression(max_iter=1000, random_state=42)

# Train it on the training data
model.fit(X_train_scaled, y_train)

print("Model trained!")
print("Intercept (bias):", model.intercept_)
print("Coefficients (weights):", model.coef_)


Model trained!
Intercept (bias): [-4.81942516]
Coefficients (weights): [[ 1.43203917 -0.96789461  2.01576993  2.7143761   0.78690264  0.33341553
   0.09277009]]


In [3]:
import pandas as pd

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Weight": model.coef_[0]
}).sort_values("Weight", key=abs, ascending=False)

print(coef_df)

                   Feature    Weight
3              Torque [Nm]  2.714376
2   Rotational speed [rpm]  2.015770
0      Air temperature [K]  1.432039
1  Process temperature [K] -0.967895
4          Tool wear [min]  0.786903
5                   Type_L  0.333416
6                   Type_M  0.092770


What your model is telling you
RankFeatureWeightWhat it means1Torque [Nm]+2.71Higher torque → strong push toward failure2Rotational speed [rpm]+2.02Higher rpm → strong push toward failure3Air temperature [K]+1.43Higher air temp → push toward failure4Process temperature [K]−0.97Higher process temp → push toward healthy5Tool wear [min]+0.79More wear → mild push toward failure6Type_L+0.33Low-quality machines → slight push toward failure (vs. High)7Type_M+0.09Medium machines ≈ same as High
Key insight
Torque and rotational speed are the two biggest drivers of failure in your model — that matches physical intuition. High torque + high spin = mechanical stress. And during EDA you saw exactly this: failures clustered at extreme torque values.
The weird one: Process temperature is negative?
That looks counterintuitive — you'd think hotter = more failure. But remember two things:

Process temperature and Air temperature are highly correlated (you saw this in your correlation heatmap). When two features are correlated, the model splits the "temperature signal" between them in odd ways — sometimes one gets a positive weight and the other gets a negative weight.
The net effect of temperature is still positive (air temp weight +1.43 beats process temp weight −0.97). The model isn't saying "heat prevents failure" — it's just bookkeeping.

This is called multicollinearity. Worth knowing the term but not a problem here because we don't care about interpreting individual weights — we care about overall prediction accuracy.

In [4]:
# Predict class labels (0 or 1) for every row in X_test
y_pred = model.predict(X_test_scaled)

# Also get the raw probabilities (useful later)
y_pred_proba = model.predict_proba(X_test_scaled)

print("First 10 predictions:", y_pred[:10])
print("First 10 actual values:", y_test.values[:10])
print("\nFirst 5 probability pairs:")
print(y_pred_proba[:5])

First 10 predictions: [1 0 0 0 0 0 0 0 0 0]
First 10 actual values: [0 0 0 0 0 0 0 0 0 0]

First 5 probability pairs:
[[3.95409506e-01 6.04590494e-01]
 [9.69402453e-01 3.05975467e-02]
 [9.48416626e-01 5.15833741e-02]
 [9.99694423e-01 3.05576594e-04]
 [8.17148452e-01 1.82851548e-01]]


In [5]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 0.9675
Accuracy: 96.75%


In [6]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1928    4]
 [  61    7]]


In [7]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=["Actual: Healthy (0)", "Actual: Failure (1)"],
    columns=["Predicted: Healthy (0)", "Predicted: Failure (1)"]
)
print(cm_df)

                     Predicted: Healthy (0)  Predicted: Failure (1)
Actual: Healthy (0)                    1928                       4
Actual: Failure (1)                      61                       7


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\n--- Full classification report ---")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Failure"]))

Precision: 0.6364
Recall:    0.1029
F1 Score:  0.1772

--- Full classification report ---
              precision    recall  f1-score   support

     Healthy       0.97      1.00      0.98      1932
     Failure       0.64      0.10      0.18        68

    accuracy                           0.97      2000
   macro avg       0.80      0.55      0.58      2000
weighted avg       0.96      0.97      0.96      2000



Your exact results
Predicted: HealthyPredicted: FailureActual: HealthyTN = 1928FP = 4Actual: FailureFN = 61TP = 7
What this literally says

1928 True Negatives — Healthy machines correctly called healthy. ✓
4 False Positives — Healthy machines wrongly flagged as failing (false alarms). Mild annoyance.
61 False Negatives — Actual failures that the model completely missed. This is the scary number.
7 True Positives — Actual failures the model correctly caught.

Let's do the math
Total test set = 1928 + 4 + 61 + 7 = 2000 ✓
Total actual failures = 61 + 7 = 68 (so there were 68 real failures in the test set)
How many did we catch? 7 out of 68.
That's the uncomfortable truth behind that "97%+ accuracy" number. We caught only 7 out of 68 failures — about 10% of the actual failures. The other 90% slipped past.
In a real factory, this model would miss 9 out of every 10 machine failures. That's terrible for predictive maintenance — the whole point was to catch failures before they happen.

Step 8: Compute precision, recall, F1 by hand
Let's calculate the real metrics using your four numbers, so you feel what they mean.
Accuracy (the misleading one)
Accuracy = (TP + TN) / Total
         = (7 + 1928) / 2000
         = 1935 / 2000
         = 0.9675 → 96.75%
Looks impressive. But...
Precision (quality of failure predictions)
"When my model says FAILURE, how often is it right?"
Precision = TP / (TP + FP)
          = 7 / (7 + 4)
          = 7 / 11
          = 0.636 → 63.6%
When your model shouts "failure!", it's correct 63.6% of the time. Not great, not terrible.
Recall (coverage of actual failures)
"Of all the actual failures, how many did my model catch?"
Recall = TP / (TP + FN)
       = 7 / (7 + 61)
       = 7 / 68
       = 0.103 → 10.3%
This is the real problem. Your model caught only 10.3% of the actual failures. A maintenance team using this model would be blindsided 9 out of 10 times.
F1 score (balance of precision and recall)
F1 combines precision and recall into a single number. It's the harmonic mean:
F1 = 2 × (Precision × Recall) / (Precision + Recall)
   = 2 × (0.636 × 0.103) / (0.636 + 0.103)
   = 2 × 0.0655 / 0.739
   = 0.131 / 0.739
   = 0.177 → 17.7%
F1 = 17.7%. That's the honest summary of your model's performance on the failure class.
Summary of your model's real performance
MetricValueWhat it meansAccuracy96.75%Misleading — mostly reflects correct healthy predictionsPrecision63.6%64% of failure alarms are realRecall10.3%Only catches 10% of actual failuresF117.7%Weighted balance — failure class performance is poor

Why this happened
The model is biased toward predicting "healthy" because 96.6% of the training data is healthy. The sigmoid threshold of 0.5 means a row needs >50% confidence before being called a failure. Since most rows have failure probabilities in the 5-20% range (you saw that in Step 5), very few cross the 50% line.
The good news: we can fix this. Two options for later:

Lower the decision threshold — instead of 0.5, use something like 0.2. The model becomes more willing to flag failure.
Use a better algorithm — Random Forest (Day 7) usually handles imbalance better.

In [9]:
# Get the failure probabilities (column 1 of predict_proba)
failure_probs = y_pred_proba[:, 1]

# Apply a custom threshold
threshold = 0.2
y_pred_custom = (failure_probs >= threshold).astype(int)

print(f"With threshold = {threshold}:")
print(f"  Predicted failures: {y_pred_custom.sum()} out of 2000")

With threshold = 0.2:
  Predicted failures: 70 out of 2000


Threshold tuning — the lever that changes everything
The concept, literally
Right now, your model converts probabilities to 0/1 predictions using this rule:
If probability of failure ≥ 0.5 → predict 1 (failure)
If probability of failure <  0.5 → predict 0 (healthy)
That 0.5 is not sacred. It's just the default. We can change it.
What happens if we lower the threshold to 0.2?
Now the rule becomes:
If probability of failure ≥ 0.2 → predict 1 (failure)
More rows will now cross the bar and be flagged as failure. This means:

More failures will be caught (recall goes UP — good!)
More false alarms will happen (precision goes DOWN — trade-off)

Why we'd accept this trade
In predictive maintenance:

Missing a failure = machine breaks unexpectedly, production halts, huge cost
False alarm = technician checks a machine that turns out to be fine, minor cost

Missing failures is much worse than false alarms. So we intentionally bias toward catching more, even at the cost of some wasted inspections. This is a real engineering decision — not just a coding trick.

In [10]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

cm_custom = confusion_matrix(y_test, y_pred_custom)
print("Confusion matrix at threshold 0.3:")
print(cm_custom)
print()
print(f"Precision: {precision_score(y_test, y_pred_custom):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_custom):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_custom):.4f}")

Confusion matrix at threshold 0.3:
[[1894   38]
 [  36   32]]

Precision: 0.4571
Recall:    0.4706
F1 Score:  0.4638


In [11]:
import joblib

# Save the trained model
joblib.dump(model, "../models/logistic_regression_model.pkl")

# Save the chosen threshold as a simple float
joblib.dump(0.2, "../models/lr_threshold.pkl")

print("Model and threshold saved.")

# Verify
import os
print("\nFiles in ../models/:")
for f in sorted(os.listdir("../models")):
    print(" ", f)

Model and threshold saved.

Files in ../models/:
  X_test_scaled.pkl
  X_train_scaled.pkl
  feature_names.pkl
  logistic_regression_model.pkl
  lr_threshold.pkl
  scaler.pkl
  y_test.pkl
  y_train.pkl


Code breakdown
joblib.dump(model, ...) — same pattern you used in Day 5 for saving the scaler. Takes the object, writes it to a file.
joblib.dump(0.2, ...) — joblib can save any Python object, including a single number. We save our chosen threshold so the dashboard uses the same one.
sorted(os.listdir("../models")) — lists the folder contents alphabetically.
Run it. You should now see eight files in ../models/:

# Day 6 — Logistic Regression Baseline

## What I did

### 1. Loaded preprocessed data from Day 5
Loaded `X_train_scaled`, `X_test_scaled`, `y_train`, `y_test`, and `feature_names` using `joblib`.

### 2. Trained a Logistic Regression model
```python
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
```
Logistic Regression computes a weighted sum of the features, passes it through a sigmoid function to get a probability between 0 and 1, and compares that probability to a threshold (default 0.5) to output a class.

### 3. Inspected learned feature weights
| Feature | Weight |
|---|---|
| Torque [Nm] | +2.71 |
| Rotational speed [rpm] | +2.02 |
| Air temperature [K] | +1.43 |
| Process temperature [K] | −0.97 |
| Tool wear [min] | +0.79 |
| Type_L | +0.33 |
| Type_M | +0.09 |

**Torque and rotational speed are the biggest failure drivers.** This matches physical intuition: high torque + high rpm = mechanical stress. The negative sign on process temperature is a side effect of its high correlation with air temperature (multicollinearity).

### 4. Evaluated on the test set (default threshold 0.5)
Confusion matrix:
|  | Pred Healthy | Pred Failure |
|---|---|---|
| **Actual Healthy** | 1928 | 4 |
| **Actual Failure** | 61 | 7 |

- Accuracy: 96.75% (misleadingly high due to class imbalance)
- Precision: 63.6%
- **Recall: 10.3%** ← Only caught 7 out of 68 real failures
- F1: 17.7%

**Lesson learned:** Accuracy is useless on imbalanced datasets. A dumb "always predict healthy" model would have scored 96.6% accuracy too.

### 5. Tuned the decision threshold
Lowered the threshold below the default 0.5 to favor catching more failures.

| Threshold | Precision | Recall | F1 |
|---|---|---|---|
| 0.5 | 63.6% | 10.3% | 17.7% |
| 0.3 | 53.5% | 33.8% | 41.4% |
| **0.2** | 45.7% | 47.1% | **46.4%** |
| 0.1 | 29.9% | 63.2% | 40.6% |

Chose **threshold = 0.2** for balanced precision/recall. In production, the final choice would depend on the cost of missed failures vs. false alarms.

### 6. Saved model + threshold
Saved `logistic_regression_model.pkl` and `lr_threshold.pkl` to `../models/` for use in the Streamlit dashboard.

## Key concepts learned today

| Concept | One-line summary |
|---|---|
| Logistic Regression | Linear model that outputs failure probability via sigmoid. |
| Weighted sum + sigmoid | z = Σ(weight × feature) + bias; probability = 1/(1+e^−z). |
| Confusion matrix | The four-cell table (TN/FP/FN/TP) — the truth about classifier performance. |
| Accuracy trap | High accuracy can hide poor failure detection on imbalanced data. |
| Precision | TP / (TP + FP) — how often a positive prediction is correct. |
| Recall | TP / (TP + FN) — how many real positives we caught. |
| F1 score | Harmonic mean of precision and recall. |
| Threshold tuning | Adjusting the 0.5 cutoff to favor recall or precision depending on use case. |
| Multicollinearity | Correlated features producing weird weight signs — cosmetic, not a performance issue. |

## Next step: Day 7
Train a **Random Forest** classifier. Expected to handle class imbalance better out of the box and potentially deliver better F1 without needing aggressive threshold tuning.